In [75]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime
import pytz

In [76]:
apps_df = pd.read_csv(
    "C:/Users/sathwika/OneDrive/Documents/Desktop/elevance_internship_training/datasets/googleplaystore.csv"
)

reviews_df = pd.read_csv(
    "C:/Users/sathwika/OneDrive/Documents/Desktop/elevance_internship_training/datasets/googleplaystore_user_reviews.csv"
)

In [77]:
merged_df = apps_df.merge(reviews_df, on="App", how="left")

In [78]:
merged_df["Reviews"] = pd.to_numeric(
    merged_df["Reviews"], errors="coerce"
)

merged_df["Rating"] = pd.to_numeric(
    merged_df["Rating"], errors="coerce"
)

merged_df["Last Updated"] = pd.to_datetime(
    merged_df["Last Updated"], errors="coerce"
)

merged_df["Installs"] = (
    merged_df["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

merged_df["Installs"] = pd.to_numeric(
    merged_df["Installs"], errors="coerce"
)

In [79]:
filtered = merged_df[
    merged_df["Reviews"] > 500
]

In [80]:
filtered = filtered[
    ~filtered["App"].str.upper().str.startswith(("X","Y","Z"))
]

In [81]:
filtered = filtered[
    ~filtered["App"].str.contains("S", case=False, na=False)
]

In [82]:
filtered = filtered[
    filtered["Category"].str.startswith(("E","C","B"))
]

In [83]:
translation = {
    "BEAUTY":"सौंदर्य",
    "BUSINESS":"வணிகம்",
    "DATING":"Dating"
}

filtered["Category"] = (
    filtered["Category"]
    .str.upper()
    .replace(translation)
)

In [84]:
filtered["Month"] = (
    filtered["Last Updated"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [85]:
grouped = (
    filtered
    .groupby(["Month","Category"])["Installs"]
    .sum()
    .reset_index()
)

In [86]:
grouped.columns

Index(['Month', 'Category', 'Installs'], dtype='object')

In [87]:
grouped.head()

,Month,Category,Installs
0,2014-01-01,COMMUNICATION,100000.0
1,2014-03-01,COMMUNICATION,100000.0
2,2014-05-01,வணிகம்,100000.0
3,2014-07-01,COMMUNICATION,400000000.0
4,2014-07-01,EDUCATION,10000.0


In [88]:
grouped = grouped.sort_values(["Category", "Month"])

grouped["Growth"] = (
    grouped.groupby("Category")["Installs"]
           .pct_change()
)

In [89]:
grouped.head(10)

,Month,Category,Installs,Growth
5,2014-10-01,BOOKS_AND_REFERENCE,500000.0,NaN
6,2014-11-01,BOOKS_AND_REFERENCE,5000000.0,9.000000
10,2015-07-01,BOOKS_AND_REFERENCE,400000000.0,79.000000
18,2016-06-01,BOOKS_AND_REFERENCE,60000.0,-0.999850
21,2016-08-01,BOOKS_AND_REFERENCE,4000000.0,65.666667
27,2017-02-01,BOOKS_AND_REFERENCE,100000.0,-0.975000
29,2017-04-01,BOOKS_AND_REFERENCE,1000000.0,9.000000
31,2017-05-01,BOOKS_AND_REFERENCE,100000.0,-0.900000
32,2017-07-01,BOOKS_AND_REFERENCE,100000.0,0.000000
33,2017-08-01,BOOKS_AND_REFERENCE,400010000.0,3999.100000


In [90]:
print(grouped.columns)

Index(['Month', 'Category', 'Installs', 'Growth'], dtype='object')


In [91]:
for cat in grouped["Category"].unique():
    temp = grouped[grouped["Category"] == cat]
    print(cat)
    print(temp.columns)
    break

BOOKS_AND_REFERENCE
Index(['Month', 'Category', 'Installs', 'Growth'], dtype='object')


In [92]:
growth = temp[temp["Growth"] > 0.20]

In [93]:
growth = temp[temp["Growth"].fillna(0) > 0.20]

In [94]:
for cat in grouped["Category"].unique():
    temp = grouped[grouped["Category"] == cat]
    print(temp.columns)
    break

Index(['Month', 'Category', 'Installs', 'Growth'], dtype='object')


In [95]:
import plotly.graph_objects as go

fig = go.Figure()

for cat in grouped["Category"].unique():

    temp = grouped[grouped["Category"] == cat].sort_values("Month")

    # Line
    fig.add_trace(
        go.Scatter(
            x=temp["Month"],
            y=temp["Installs"],
            mode="lines+markers",
            name=cat
        )
    )

    # Highlight growth > 20%
    growth = temp[temp["Growth"].fillna(0) > 0.20]

    if not growth.empty:
        fig.add_trace(
            go.Scatter(
                x=growth["Month"],
                y=growth["Installs"],
                fill="tozeroy",
                mode="none",
                showlegend=False
            )
        )

fig.update_layout(
    title="Total Installs Over Time by Category",
    xaxis_title="Month",
    yaxis_title="Total Installs",
    template="plotly_white"
)

fig.show()

In [96]:
import plotly.graph_objects as go

grouped = grouped.sort_values(["Category", "Month"])

fig = go.Figure()

for cat in grouped["Category"].unique():

    temp = grouped[grouped["Category"] == cat].copy()
    fig.add_trace(
        go.Scatter(
            x=temp["Month"],
            y=temp["Installs"],
            mode="lines+markers",
            name=cat,
            line=dict(width=3),
            marker=dict(size=7),
            hovertemplate=
            "<b>%{fullData.name}</b><br>"
            "Month: %{x|%b %Y}<br>"
            "Installs: %{y:,.0f}<extra></extra>"
        )
    )
    growth = temp[temp["Growth"].fillna(0) > 0.20]

    if not growth.empty:

        # Shade Area
        fig.add_trace(
            go.Scatter(
                x=growth["Month"],
                y=growth["Installs"],
                mode="lines",
                line=dict(width=0),
                fill="tozeroy",
                fillcolor="rgba(50,205,50,0.25)",
                showlegend=False,
                hoverinfo="skip"
            )
        )

        # Diamond Marker
        fig.add_trace(
            go.Scatter(
                x=growth["Month"],
                y=growth["Installs"],
                mode="markers",
                marker=dict(
                    symbol="diamond",
                    size=12,
                    color="red"
                ),
                showlegend=False,
                hovertemplate=
                "<b>Growth > 20%</b><br>"
                "Month: %{x|%b %Y}<br>"
                "Installs: %{y:,.0f}<extra></extra>"
            )
        )

fig.update_layout(

    title={
        "text":"<b>Google Play Store - Total Installs Trend by Category</b>",
        "x":0.5,
        "xanchor":"center"
    },
    xaxis_title="Month",
    yaxis_title="Total Installs",
    template="plotly_white",
    hovermode="x unified",
    legend_title="Category",
    width=1200,
    height=650
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="lightgray"
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray",
    tickformat=","
)

fig.show()

In [97]:
from datetime import datetime
import pytz

ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

start_time = datetime.strptime("18:00", "%H:%M").time()
end_time = datetime.strptime("21:00", "%H:%M").time()

if start_time <= current_time <= end_time:
    fig.show()
else:
    print("This graph is available only between 6:00 PM and 9:00 PM IST.")